In [ ]:
import os
import fm
import argparse
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

In [2]:
env_path = Path(os.getcwd()).parent  /'Config' / '.env'

load_dotenv(env_path)
data_path = os.getenv('DATA_PATH')
CACHE_PATH = os.getenv('CACHE_PATH')


hu_path = data_path + 'Hu.csv'
mix_path = data_path + 'Mix.csv'
taka_path = data_path + 'Taka.csv'

hu_df = pd.read_csv(hu_path)
mix_df = pd.read_csv(mix_path)
taka_df = pd.read_csv(taka_path)

print(hu_df.shape, mix_df.shape, taka_df.shape)

(2361, 5) (472, 5) (702, 5)


In [3]:
combined_df = pd.concat([hu_df, mix_df, taka_df], axis=0)
combined_df = combined_df.sample(frac=1)
print(combined_df.shape)

(3535, 5)


In [4]:
combined_df.head()

,siRNA,mRNA,label,y,td
322,GCGGGUAACCCUGGCCCAC,GUUUGUGUUCAGUUUUAAGGUGGGCCAGGGUUACCCGCAUGAUCCC...,0.420582,0,"-0.73,-3.42,-14.88,0,1,-210.59,0.1052631578947..."
210,UCCUUUCUCUCCUGUAGCU,UGGAUAAUUUUGUGGCCUUAGCUACAGGAGAGAAAGGAUUUGGCUA...,0.836834,1,"-0.27,-2.35,-12.44,1,0,-186.42,0.4736842105263..."
783,AUGGUCAGCAGUACGUGUC,ACGGCUGAGCUGGGCAUCCGACACGUACUGCUGACCAUCAAGUGCC...,0.637584,1,"1.7,-1.1,-9.38,0,0,-193.25,0.2631578947368421,..."
277,CACAGGUGGGUCCUCUUGU,UGCGGGAUUUCAAGCGGUUACAAGAGGACCCACCUGUGGGUGUCAG...,0.455630,0,"-0.77,-2.11,-10.44,0,0,-196.73,0.3157894736842..."
413,CAAAAUGCUUCUUGAUCUU,CAAAUAUGCAGAGUCAGAAAAGAUCAAGAAGCAUUUUGAGGAAGCC...,0.460104,0,"-1.63,-2.11,-10.44,0,0,-167.30999999999997,0.4..."


In [5]:
print(list(map(float, combined_df['td'].iloc[1].strip().split(','))))

[-0.27, -2.35, -12.44, 1.0, 0.0, -186.42, 0.4736842105263158, 0.0, 0.10526315789473684, 0.0, 0.0, 0.0, -3.26, 0.05555555555555555, 0.0, 0.0, 0.1111111111111111, -2.08, 0.0, 0.05555555555555555, 0.0, -2.11, 0.1111111111111111, 0.0]


In [6]:
col_names = [
    "delta_G_end",                # ΔG°end: End free energy difference
    "delta_G1",                   # ΔG1: Free energy change of the first base pair
    "delta_H1",                   # ΔH1: Enthalpy change of the first base pair
    "U1",                         # U1: Uracil content of the first base pair
    "G1",                         # G1: Guanine content of the first base pair
    "delta_H_all",               # ΔHall: Total enthalpy change of all base pairs
    "U_all",                      # Uall: Total uracil content of all base pairs
    "U1_neighbors",              # U1U1: Uracil content of first base pair and its neighbors
    "G_all",                      # Gall: Total guanine content of all base pairs
    "GC1",                        # GC1: Guanine and cytosine content of the first base pair
    "GG1_neighbors",            # GG1: Guanine content of the first base pair and its neighbors
    "GC_all",                     # GCall: Total guanine and cytosine content of all base pairs
    "delta_G2",                  # ΔG2: Free energy change of the second base pair
    "UA_all",                     # UAall: Total uracil and adenine content of all base pairs
    "U2",                         # U2: Uracil content of the second base pair
    "C1",                         # C1: Cytosine content of the first base pair
    "C_all",                      # Ccall: Total cytosine content of all base pairs
    "delta_G18",                # ΔG18: Free energy change of the eighteenth base pair
    "GC1_neighbors",           # GC1: Cytosine content of the first base pair and its neighbors
    "GC_total",                  # GCall: Total guanine and cytosine content of all base pairs
    "GC1_gc",                     # GC1: Guanine and cytosine content of the first base pair
    "delta_G13",                # ΔG13: Free energy change of the thirteenth base pair
    "UA_all_2",                  # UAall: Uracil and adenine content of all base pairs (again)
    "A19"                         # A19: Adenine content of the nineteenth base pair
]

# --------------------------------------------------------------------
# OPTION A – one-liner, stays vectorised (recommended once you trust it)
# --------------------------------------------------------------------
# In pandas every column is a Series object.
# If the series holds text (dtype == object or the new pandas "string" dtype), pandas adds a special accessor called .str that gives you dozens of 
# vectorised string methods

clean = (
    combined_df["td"]              # start from the td column
      .str.strip()                 # drop leading/trailing spaces/newlines
      .str.replace(" ", "", regex=False)  # erase spaces *inside* the string
)



#   clean.str.split(",", expand=True) Splits every string on commas at once, returning a DataFrame (because expand=True).
#   And the result is 24 columns of strings
#   Then .astype(float) converts each column to float64 in one vectorised shot.

split_df = clean.str.split(",", expand=True).astype(float)
split_df.columns = col_names

# drop the old column and stick the new ones in
combined_df.rename(columns={'siRNA':'Antisense', 'mRNA':'mrna'}, inplace=True)
combined_df = pd.concat([combined_df.drop(columns="td"), split_df], axis=1)

In [7]:
combined_df.head()

,Antisense,mrna,label,y,delta_G_end,delta_G1,delta_H1,U1,G1,delta_H_all,...,U2,C1,C_all,delta_G18,GC1_neighbors,GC_total,GC1_gc,delta_G13,UA_all_2,A19
1356,AUCUGCAGUGCUGGGGACC,GAUAUUUUGAAAGAUAAGUGGUCCCCAGCACUGCAGAUCCGCACAG...,0.463833,0,2.16,-1.10,-9.38,0.0,0.0,-206.25,...,1.0,0.0,0.055556,-3.26,0.0,0.111111,0.0,-3.26,0.000000,0.0
121,UUUCCGUCAUCGUCUUUCC,CGUUGUUGUUUUGGAGCACGGAAAGACGAUGACGGAAAAAGAGAUC...,0.932149,1,2.33,-0.93,-6.82,1.0,0.0,-183.31,...,1.0,0.0,0.111111,-3.26,0.0,0.000000,0.0,-2.35,0.222222,0.0
1219,UAAAUGGGUAUGUUUCUGG,CCAACUAGAGAUAAAAAUACCAGAAACAUACCCAUUUAAUCCCCCU...,0.539896,1,1.93,-1.33,-7.69,1.0,0.0,-171.30,...,0.0,0.0,0.000000,-3.26,0.0,0.000000,0.0,-0.93,0.111111,0.0
543,CACAAAGCAGAGGUUAGCC,AGCUGGAGCUGCCUCAAGAGGCUAACCUCUGCUUUGUGGACAUUGA...,0.523490,1,0.70,-2.11,-10.44,0.0,0.0,-189.56,...,0.0,1.0,0.055556,-3.26,0.0,0.111111,0.0,-2.24,0.055556,0.0
1684,CUGAAUCUCCAACUGCAAA,CACCAGUUUAGAAGGUCCGUUUGCAGUUGGAGAUUCAGAUUGCCUC...,0.362416,0,-1.60,-2.08,-10.48,0.0,0.0,-179.52,...,1.0,1.0,0.055556,-0.93,0.0,0.055556,0.0,-2.08,0.000000,1.0


## RNA-FM

In [ ]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)


# !nvidia-smi

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


def pad_and_stack(list_of_ndarrays):
    """
    Convert a list of [L_i, D] numpy arrays (variable‐length sequences)
    into a single tensor of shape [B, L_max, D] with zero-padding.
    """
    tensors = [torch.as_tensor(x, dtype=torch.float32) for x in list_of_ndarrays]
    return pad_sequence(tensors, batch_first=True)          # [B, L_max, D]




#   Code for Embeding generation and Integration
def check_model_cache(CACHE_PATH) -> str:
    """
    Check if RNA-FM model is available in cache.

    Returns:
     Path of the model stored in cache (so that we dont need to download everytime)
    """

    cache_path = Path.home() / CACHE_PATH
    
    if cache_path.exists():
        return str(cache_path)
    else:
        return None
   



def load_rna_fm_fast(model_path=None):
    torch.serialization.add_safe_globals([argparse.Namespace])

    if model_path and os.path.exists(model_path):
        model, alphabet = fm.pretrained.load_model_and_alphabet_local(
            model_path,
            theme="rna"
        )
    else:
        print("Model Downloading....")
        model, alphabet = fm.pretrained.rna_fm_t12()

    batch_converter = alphabet.get_batch_converter()
    return model, alphabet, batch_converter




def generate_embeddings(seq_list, model, alphabet, batch_converter):
    modified_seq = []
    for i, seq in enumerate(seq_list, 1):
        modified_seq.append( (str(i), seq ))

    model.eval()
    batch_labels, batch_strs, batch_tokens = batch_converter(modified_seq)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[12])

    token_embeddings = results["representations"][12]
    token_embeddings_np = token_embeddings.cpu().numpy()
    num_seq = token_embeddings.shape[0]

    final_embeddings = []
    for i in range(num_seq):
        final_embeddings.append(token_embeddings_np[i])

    return final_embeddings




def load_RNAFM_and_data(data, CACHE_PATH):    

    # data, UP_LEN, SI_LEN, DOWN_LEN, MRNA_LEN = input_to_inference(data_pre, mRNA_seq)

    cache_model_path = check_model_cache(CACHE_PATH)
    model, alphabet, batch_converter = load_rna_fm_fast(cache_model_path)
    
    model.eval()
    # sense_seq = data['Sense'].to_list()
    siRNA_seq, mRNA_seq = data['Antisense'].to_list(), data['mrna'].to_list()
    

    siRNA_embeddings = generate_embeddings(siRNA_seq, model, alphabet, batch_converter)
    mRNA_embeddings = generate_embeddings(mRNA_seq, model, alphabet, batch_converter)

    return siRNA_seq, siRNA_embeddings, mRNA_embeddings # UP_LEN, SI_LEN, DOWN_LEN, MRNA_LEN

cuda


In [8]:
#   Working perfectly fine...
#   Takes around 1 min to process all embeddings

siRNA_seq, siRNA_embeddings, mRNA_embeddings = load_RNAFM_and_data(combined_df, CACHE_PATH)

In [9]:
print(siRNA_embeddings[5].shape)
print(mRNA_embeddings[2].shape)

(21, 640)
(59, 640)


# Model Coding:-

## ConvNetXt 

### Some technicalities of ConvNetXt directly from paper

- Downsampling is achieved by setting the **convolutional stride** to 2 at the end of the first two **stages**
- The convolutional operations are divided into three stages, with downsampling achieved by setting the convolutional stride to 2 at the end of the first two stages
- The first convolutional **stage** reduces the hidden layer dimension from 128 to 64, followed by a stride-2 convolutional layer for further downsampling. This stage contains 2 ConvNeXtBlocks
- The second convolutional stage reduces the hidden layer dimension from 64 to 32, employing multiple stride-1 and stride-2 convolutional layers to gradually extract features. This stage comprises 3 ConvNeXtBlocks
- The final convolutional stage maintains the dimension at 32 and includes 1 ConvNeXtBlock

*After the convolutional stages, a flattening operation merges the hidden and sequence dimensions before passing the features to the task-specific layers*

In [38]:
class ConvNetXtBlock(nn.Module):
    def __init__(
            self,
            kernel_size_pool=3,
            stride_pool=1,
            in_channels=128,
            out_channels=64,
            kernel_size_conv=7,
            stride_conv=2,
            embedding_dim_1=None,
            embedding_dim_2=None,
            in_features=None,
            out_features=None,
            dropout=0.1
        ):
    
        super().__init__()

        embedding_dim_1 = embedding_dim_1 or out_channels
        embedding_dim_2 = embedding_dim_2 or out_channels
        in_features = in_features or out_channels
        out_features = out_features or out_channels * 4

        
        self.pool = (
            nn.AvgPool1d(kernel_size_pool, stride_pool)
            if kernel_size_pool is not None
            else nn.Identity()
        )
    
        self.conv = nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size_conv, stride=stride_conv, padding=kernel_size_conv // 2, groups=out_channels)
        self.norm1 = nn.LayerNorm(embedding_dim_1)
        self.ffn    = nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(out_features, in_features),
        )
        self.norm2 = nn.LayerNorm(embedding_dim_2)


        needs_proj = (stride_conv != 1) or (in_channels != out_channels)
        self.skip_proj = (
            nn.Sequential(
                self.pool,                                 # same pool
                nn.Conv1d(in_channels, out_channels, 1,    # 1×1 conv
                          stride=stride_conv)
            ) if needs_proj else nn.Identity()
        )
        


    def forward(self, x):
        skip = self.skip_proj(x)
        print(x.shape)

        x = self.pool(x)

        print(x.shape)

        x = self.conv(x)

        print(x.shape)

        x = x.transpose(1,2)   #    Transposing the 1st and 2nd dim (Needed to maintain dim uniformity)

        print(x.shape)

        x = self.norm1(x)

        print(x.shape)

        x = self.ffn(x)

        print(x.shape)

        output = self.norm2(x + skip.transpose(1,2))     # Initially x has 640 cols, so res also must have 640 cols/features

        print('-'*35)

        return output.transpose(1,2)  #     Back to the original dimension


In [ ]:
class ConvNetXtEncoder(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()

        #   128 -> 64
        self.convnet_block_1a = ConvNetXtBlock(stride_conv=1, in_channels=128, out_channels=64, dropout=dropout, kernel_size_pool=None)
        self.convnet_block_1b = ConvNetXtBlock(stride_conv=2, in_channels=64, out_channels=64, dropout=dropout)

        #   64 -> 32
        self.convnet_block_2a = ConvNetXtBlock(stride_conv=1, in_channels=64, out_channels=32, dropout=dropout, kernel_size_pool=None)
        self.convnet_block_2b = ConvNetXtBlock(stride_conv=1, in_channels=32, out_channels=32, dropout=dropout, kernel_size_pool=None)
        self.convnet_block_2c = ConvNetXtBlock(stride_conv=2, in_channels=32, out_channels=32, dropout=dropout)

        #   32 -> 1
        self.convnet_block_3 = ConvNetXtBlock(stride_conv=1, in_channels=32, out_channels=32, dropout=dropout, kernel_size_pool=None)

        self.gap = nn.AdaptiveAvgPool1d(1)    # Does (B, C, 1)

        #   For regression
        self.reg_head = nn.Sequential(
            nn.Flatten(1),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

        #   For classification
        self.clas_head = nn.Sequential(
            nn.Flatten(1),
            nn.Dropout(dropout),
            nn.Linear(32, 2),
            nn.Softmax(dim=1)  #    Need to cross-verify (paper dont have mention of this)
        )


    def forward(self, x):
        #   First stage
        res_1 = self.convnet_block_1a(x)
        res_1 = self.convnet_block_1b(res_1)

        #   Second stage
        res_2 = self.convnet_block_2a(res_1)
        res_2 = self.convnet_block_2b(res_2)
        res_2 = self.convnet_block_2c(res_2)

        #   Third stage
        outputs = self.convnet_block_3(res_2)


        fin_op = self.gap(outputs)  #   (B, 32, 1)
        fin_op = fin_op.flatten(1)  #   (B, 32)

        y_reg = self.reg_head(fin_op)
        y_cls = self.clas_head(fin_op)

        return y_reg.squeeze(-1), y_cls


        #   Flatten
        # outputs = outputs.transpose(1, 2).contiguous().view(outputs.size(0), -1)

        # return outputs

#### Some dummy experiments with the blocks

In [40]:
B = 64
c = ConvNetXtBlock(                    # let the defaults work for you
    stride_conv=2,                         # keep the down-sample
    in_channels=128,
    out_channels=64
)                                          # ← DON’T override embedding_dim_*
x = torch.rand(B, 128, 640)

ans = c(x)
print(ans.shape)

torch.Size([64, 128, 640])
torch.Size([64, 128, 638])
torch.Size([64, 64, 319])
torch.Size([64, 319, 64])
torch.Size([64, 319, 64])
torch.Size([64, 319, 64])
-----------------------------------
torch.Size([64, 64, 319])


In [41]:
my_net = ConvNetXtEncoder()
B = 64
input_emb = torch.rand(B, 128, 1024)

output_r, output_c = my_net(input_emb)
print(output_r.shape, output_c.shape)

torch.Size([64, 128, 1024])
torch.Size([64, 128, 1024])
torch.Size([64, 64, 1024])
torch.Size([64, 1024, 64])
torch.Size([64, 1024, 64])
torch.Size([64, 1024, 64])
-----------------------------------
torch.Size([64, 64, 1024])
torch.Size([64, 64, 1022])
torch.Size([64, 64, 511])
torch.Size([64, 511, 64])
torch.Size([64, 511, 64])
torch.Size([64, 511, 64])
-----------------------------------
torch.Size([64, 64, 511])
torch.Size([64, 64, 511])
torch.Size([64, 32, 511])
torch.Size([64, 511, 32])
torch.Size([64, 511, 32])
torch.Size([64, 511, 32])
-----------------------------------
torch.Size([64, 32, 511])
torch.Size([64, 32, 511])
torch.Size([64, 32, 511])
torch.Size([64, 511, 32])
torch.Size([64, 511, 32])
torch.Size([64, 511, 32])
-----------------------------------
torch.Size([64, 32, 511])
torch.Size([64, 32, 509])
torch.Size([64, 32, 255])
torch.Size([64, 255, 32])
torch.Size([64, 255, 32])
torch.Size([64, 255, 32])
-----------------------------------
torch.Size([64, 32, 255])
torc

## Transformer Block

### Some technical details from the paper:-

Transformer Encoder: This part of the model integrates global sequence information from siRNA using the T5 Encoder architecture. The configuration includes a hidden layer dimension of 128, 4 Transformer encoder layers, 4 attention heads, and a feed-forward network dimension of 128 × 4

In [42]:
#   The architecture is correct, but dimension (in-features, out-features etc) need to be adjusted.....!!

class TransformerEncoderBlock(nn.Module):
    def __init__(
            self,
            d_model=128,
            nhead=4,
            dim_feedforward=128*4,
            dropout=0.1,
        ):
        super().__init__()
        self.layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        # self.layer2 = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        # self.layer3 = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
        # self.layer4 = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout, activation='gelu', batch_first=True, norm_first=True)
    
    def forward(self, x, mask=None):
        return self.layer(x, src_key_padding_mask=mask)
    


class TransformerEncoder(nn.Module):
    def __init__(
            self,
            in_dim=640,          # dimension of incoming features per token
            num_layers=4,
            d_model=128,
            nhead=4,
            dim_feedforward=128*4,
            dropout=0.1
        ):
        super().__init__()

        self.in_proj = nn.Linear(in_dim, d_model) if in_dim != d_model else nn.Identity()
        self.encoder_layers = nn.ModuleList()
        
        # self.encoder_layers.append(
        #     TransformerEncoderBlock(
        #         d_model=640,
        #         nhead=nhead,
        #         dim_feedforward=dim_feedforward,
        #         dropout=dropout
        #     )
        # )
        self.encoder_layers.extend([
            TransformerEncoderBlock(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)


    def forward(self, x, mask=None):
        x = self.in_proj(x)           # (B, L, d_model)
        for layer in self.encoder_layers:
            x = layer(x, mask)
        x = self.norm(x)              # (B, L, d_model)
        return x

In [43]:
x = torch.tensor(siRNA_embeddings[5])
x = x.unsqueeze(0)               #  Just for now....During batch processing, the 1st dimension would be replaced by B
# x.shape

model_trans = TransformerEncoder()
model_conv = ConvNetXtEncoder()
ans_trans = model_trans(x)

print(ans_trans.shape)

new_input = ans_trans.transpose(1, 2).contiguous()
print(new_input.shape)

ans_f_reg, ans_f_cls = model_conv(new_input)
print(ans_f_reg)
print(ans_f_cls)

torch.Size([1, 21, 128])
torch.Size([1, 128, 21])
torch.Size([1, 128, 21])
torch.Size([1, 128, 21])
torch.Size([1, 64, 21])
torch.Size([1, 21, 64])
torch.Size([1, 21, 64])
torch.Size([1, 21, 64])
-----------------------------------
torch.Size([1, 64, 21])
torch.Size([1, 64, 19])
torch.Size([1, 64, 10])
torch.Size([1, 10, 64])
torch.Size([1, 10, 64])
torch.Size([1, 10, 64])
-----------------------------------
torch.Size([1, 64, 10])
torch.Size([1, 64, 10])
torch.Size([1, 32, 10])
torch.Size([1, 10, 32])
torch.Size([1, 10, 32])
torch.Size([1, 10, 32])
-----------------------------------
torch.Size([1, 32, 10])
torch.Size([1, 32, 10])
torch.Size([1, 32, 10])
torch.Size([1, 10, 32])
torch.Size([1, 10, 32])
torch.Size([1, 10, 32])
-----------------------------------
torch.Size([1, 32, 10])
torch.Size([1, 32, 8])
torch.Size([1, 32, 4])
torch.Size([1, 4, 32])
torch.Size([1, 4, 32])
torch.Size([1, 4, 32])
-----------------------------------
torch.Size([1, 32, 4])
torch.Size([1, 32, 4])
torch.S

In [44]:
ans_f_reg.item()

0.19436465203762054